# Finetuning - QLoRA

https://huggingface.co/docs/peft/en/developer_guides/quantization


## Quantization
4비트 양자화(4-bit Quantization)는 모델의 가중치를 정밀도가 낮은 4비트 데이터 형식으로 변환하여 메모리 사용량을 획기적으로 줄이는 기술이다. 지적한 대로 모든 파라미터가 양자화 대상이 되는 것은 아니며, 성능 유지를 위해 전략적으로 적용된다.
4비트 양자화는 **"대세인 가중치는 작게 줄이고, 민감한 레이어와 통계 정보는 원본을 유지"**하는 전략이다. 이를 통해 일반 소비자용 GPU(예: RTX 3090/4090 24GB)에서도 30B급 대형 모델을 구동할 수 있게 된다.


**1. 4비트 양자화 시 메모리 변화**

30B 파라미터 모델을 기준으로 계산하면 다음과 같은 변화가 발생한다.

- **BF16 (기본):** 파라미터당 2바이트 $\rightarrow$ 약 **60GB** 필요
- **4-bit (양자화):** 파라미터당 0.5바이트 $\rightarrow$ 약 **15GB** 필요 (이론상 1/4 수준)

실제로는 양자화 과정에서 발생하는 스케일링 계수(Scaling Factor)와 메타데이터 때문에 약 **17~18GB** 정도의 VRAM을 사용하게 된다.

**2. 왜 모든 파라미터를 양자화하지 않는가?**

모델의 성능(Perplexity) 저하를 최소화하기 위해 **혼합 정밀도(Mixed Precision)** 방식을 사용한다.

- **양자화 대상 (Linear Layers):** 모델의 대부분을 차지하는 행렬 연산 가중치(Attention, MLP 레이어 등)는 4비트로 변환하여 용량을 줄인다.
- **양자화 제외 (Sensitive Layers):**
    - **Normalization 레이어:** LayerNorm 등은 수치 민감도가 매우 높아 원본 정밀도(FP32/BF16)를 유지한다.
    - **Embedding 레이어:** 텍스트를 벡터로 변환하는 첫 단계이므로 정밀도가 중요하다.
    - **LM Head:** 최종 출력층은 예측 정확도를 위해 보통 양자화하지 않는다.

**3. 주요 양자화 기법 (NF4)**

단순히 소수점을 자르는 것이 아니라, 데이터의 분포를 고려한 알고리즘을 사용한다. 가장 대표적인 것이 **NF4(NormalFloat 4)**이다.

- **특징:** 가중치가 정규분포를 따른다는 가정하에, 값이 몰려 있는 구간에는 촘촘하게, 값이 적은 구간에는 넓게 비트를 할당한다.
- **장점:** 일반적인 4비트 정수형(Int4)보다 정보 손실이 훨씬 적어 모델의 추론 능력을 잘 보존한다.

In [ ]:
!pip install transformers datasets accelerate peft bitsandbytes hf_transfer wandb trl

   ---------------------------------------- 0.0/39.1 MB ? eta -:--:--
   -- ------------------------------------- 2.1/39.1 MB 13.0 MB/s eta 0:00:03
   ------ --------------------------------- 6.6/39.1 MB 18.3 MB/s eta 0:00:02
   ------------- -------------------------- 12.8/39.1 MB 22.3 MB/s eta 0:00:02
   ---------------- ----------------------- 16.0/39.1 MB 21.9 MB/s eta 0:00:02
   ---------------- ----------------------- 16.5/39.1 MB 17.0 MB/s eta 0:00:02
   -------------------------- ------------- 25.7/39.1 MB 21.4 MB/s eta 0:00:01
   ----------------------------------- ---- 34.3/39.1 MB 24.2 MB/s eta 0:00:01
   ---------------------------------------  39.1/39.1 MB 25.6 MB/s eta 0:00:01
   ---------------------------------------  39.1/39.1 MB 25.6 MB/s eta 0:00:01
   ---------------------------------------- 39.1/39.1 MB 20.7 MB/s  0:00:01
   ---------------------------------------- 0.0/24.1 MB ? eta -:--:--
   ----------- ---------------------------- 7.1/24.1 MB 36.4 MB/s eta 0:00:

In [2]:
!nvidia-smi

'nvidia-smi'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
HF_TOKEN = os.getenv('HF_TOKEN')

In [2]:
# import os

# HF_TOKEN = os.environ['HF_TOKEN']

## 데이터셋 로드
https://huggingface.co/datasets/capybaraOh/naver-economy-news2stock

In [3]:
from datasets import load_dataset  # HuggingFace 데이텃세 로더

# HuggingFace Hub train split 로드
dataset = load_dataset('capybaraOh/naver-economy-news2stock', split='train')
print(len(dataset))
dataset  # Dataset 객체 정보

c:\Users\playdata2\LLM\llm_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1000


Dataset({
    features: ['system', 'user', 'assistant'],
    num_rows: 1000
})

In [4]:
dataset[0]

{'system': "\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n",
 'user': '추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대

In [5]:
# HuggingFace Dataset을 학습 / 평가 분리 후 Chat 메시지 포맷으로 변환
test_ratio = 0.2  # 평가셋 비율

train_data = []  # 학습 데이터 리스트
test_data = []   # 평가 데이터 리스트

data_indices = list(range(len(dataset)))  # 전체 인덱스
test_size = int(len(dataset) * test_ratio)  # 평가셋 크기

test_data_indices = data_indices[:test_size]   # 앞부분은 평가셋 (인덱스)
train_data_indices = data_indices[test_size:]  # 나머지는 학습셋 (인덱스)

# OpenAI / Chat 학습용 포맷 : {'messages': [{system}, {user}, {assistant}]}
def format_data(data):
    return {
        'messages': [
            {
                'role': 'system',
                'content': data['system']
            },
            {
                'role': 'user',
                'content': data['user']
            },
            {
                'role': 'assistant',
                'content': data['assistant']
            }
        ]
    }

train_data = [format_data(dataset[i]) for i in train_data_indices]  # 학습 인덱스 -> 학습 dict
test_data = [format_data(dataset[i]) for i in test_data_indices]    # 평가 인덱스 -> 평가 dict

print(len(train_data))
print(len(test_data))

800
200


In [7]:
from datasets import Dataset

train_data = Dataset.from_list(train_data)
test_data = Dataset.from_list(test_data)

train_data[256]

{'messages': [{'role': 'system',
   'content': "\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n"},
  {'role': 'user',
   'content': '국토부 세종시 부적격 당첨자 철저히 조사해 엄중 조치\n원희룡 계약취소 및 주택환수 형사고발까지 필요 조치 취할 것 원희룡 국토교통부 장관. 2022.7.6 뉴스1 © News1 김명섭 기자 서울 뉴스1 박승희 기자 국토교통부는 감사원 감사에서 세종시 무자격 특공 당첨자가 무더기로 적발된 것과 관련해 위법행위 여부를 철저히 조사해 위법성이 밝혀지는 경우 엄중히 조치할 것 이라고 6일 밝혔다. 감사원은 세종시 이전기관 종사자 주택 특별공급에 대한 감사를 진행한 결과 총 45건 76명 에 대한 감사 결과를 확정했다. Δ대상관리 부실 Δ확인서 부당 발급 Δ확인서 위조 Δ중복 당첨 Δ지도·감독 소홀 등 사유로 무자격자가 대거 당첨된 것으로 드러났다. 국토부는 감사원으로부터 결과를 통보 받았으며 이에 따른 

## BaseModel + Quantizaton-Config

`BitsAndBytesConfig`는 Hugging Face Transformers에서 대형 모델을 8비트 또는 4비트로 양자화(quantization)하여 메모리 사용량을 줄이고, 저사양 환경에서도 대형 모델을 사용할 수 있게 도와주는 설정 클래스이다.

**주요 파라미터 목록**

| 파라미터명                  | 설명                                                                                          | 예시 값            |
|-----------------------------|----------------------------------------------------------------------------------------------|-------------------|
| `load_in_8bit`              | 8비트 양자화 활성화 여부. True로 설정 시 8비트로 모델 로드.                                    | True, False       |
| `load_in_4bit`              | 4비트 양자화 활성화 여부. True로 설정 시 4비트로 모델 로드.                                    | True, False       |
| `bnb_4bit_quant_type`       | 4비트 양자화 타입. `nf4`(NormalFloat4, 기본값), `fp4` 중 선택.                                 | "nf4", "fp4"      |
| `bnb_4bit_compute_dtype`    | 연산에 사용할 데이터 타입. 보통 `torch.float16`, `torch.bfloat16`, `torch.float32` 중 선택.     | torch.bfloat16    |
| `bnb_4bit_use_double_quant` | 이중 양자화 사용 여부. True로 설정 시 추가 양자화로 메모리 절감 가능.                           | True, False       |
| `llm_int8_threshold`        | 8비트 양자화 시 threshold 지정. 값이 낮을수록 더 많은 파라미터가 8비트로 변환됨.                | 0.0 ~ 6.0         |
| `llm_int8_skip_modules`     | 양자화에서 제외할 모듈 리스트.                                                                | ["lm_head"]       |
| `bnb_4bit_quant_storage`    | 4비트 파라미터 저장에 사용할 타입. 기본값은 `torch.uint8`.                                    | torch.uint8       |


- **load_in_8bit**  
  8비트 양자화를 활성화하는 플래그이다. True로 설정 시 모델 파라미터를 8비트 정수로 변환하여 메모리 사용량을 약 75%까지 줄일 수 있다.

- **load_in_4bit**  
  4비트 양자화를 활성화하는 플래그이다. True로 설정 시 더욱 극적인 메모리 절감 효과를 볼 수 있다. 4비트 양자화는 QLoRA 등 최신 연구에서 자주 사용된다.

- **bnb_4bit_quant_type**  
  4비트 양자화 시 사용할 데이터 타입을 지정한다.  
  - `nf4`: NormalFloat4 (기본값, QLoRA에서 주로 사용)  
  - `fp4`: FP4 타입.

- **bnb_4bit_compute_dtype**  
  연산(Forward/Backward) 시 사용할 데이터 타입을 지정한다.  
  - `torch.float16`, `torch.bfloat16`, `torch.float32` 등이 있다.  
  - 16비트 타입을 사용하면 연산 속도가 빨라지고, 메모리 사용량도 줄일 수 있다.

- **bnb_4bit_use_double_quant**  
  이중 양자화(nested quantization)를 활성화하는 옵션이다. True로 설정 시 한 번 더 양자화를 적용하여 메모리 사용량을 추가로 절감할 수 있다. 메모리 부족 시 유용하다.

- **llm_int8_threshold**  
  8비트 양자화 시 threshold 값을 조정하여, threshold 이하의 weight만 8비트로 변환한다. 값이 낮을수록 더 많은 파라미터가 8비트로 변환된다.

- **llm_int8_skip_modules**  
  양자화에서 제외할 모듈(레이어) 리스트를 지정한다. 예를 들어, 출력 레이어(`lm_head`) 등은 양자화에서 제외할 수 있다.

- **bnb_4bit_quant_storage**  
  4비트 파라미터 저장에 사용할 데이터 타입을 지정한다. 기본값은 `torch.uint8`이다.


**활용 팁**

- **메모리가 부족하다면**: `bnb_4bit_use_double_quant=True`로 설정.
- **정밀도가 중요하다면**: `bnb_4bit_quant_type="nf4"`로 설정.
- **학습 속도가 중요하다면**: `bnb_4bit_compute_dtype`를 16비트(float16, bfloat16)로 설정.

- `BitsAndBytesConfig`는 4비트/8비트 양자화 옵션을 통합 관리하며, 파라미터 조합을 통해 다양한 하드웨어 환경에 맞는 최적화가 가능하다.

In [8]:
from transformers import BitsAndBytesConfig
import torch

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM  # 토크나이저 / 생성형 모델 자동 로더
import torch

pretrained_model_name = 'NCSOFT/Llama-VARCO-8B-Instruct'  # 사전학습 모델명

model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name,
    dtype = torch.bfloat16,  # 가중치 로딩 dtype(bf16)
    device_map = 'auto',      # 환경에 맞춰 CPU / GPU 자동 배치
    quantization_config=quant_config
)

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name)  # 해당 모델의 토크나이저 로드

c:\Users\playdata2\LLM\llm_venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\playdata2\.cache\huggingface\hub\models--NCSOFT--Llama-VARCO-8B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 4 files:  25%|██▌       | 1/4 [04:34<13:43, 274.43s/it]


KeyboardInterrupt: 

In [ ]:
text = tokenizer.apply_chat_template(train_data[256]['messages'], tokenize=False)
print(text)

In [ ]:
# Lllam3 계열 Chat 학습용 데이터 콜레이터 : 프롬프트 생성 -> 토크나이즈/패딩 -> assistant 구간만 라벨링
def data_collator(batch, tokenizer=tokenizer, max_length=8192):
    # 1. 프롬프트 생성
    prompts = []
    for example in batch:
        prompt = '<|begin_of_text|>'
        for msg in example['messages']:
            role = msg['role']
            content = msg['content'].strip()
            prompt += f'<|start_header_id|>{role}<|end_header_id|>\n{content}<|eot_id|>'
        prompts.append(prompt)

    # 2. 토큰처리 / 패딩 / 텐서 변환
    tokenized = tokenizer(
        prompts,
        truncation=True,
        max_length=max_length,
        padding=True,
        return_tensors='pt'
    )
    input_ids = tokenized['input_ids']
    attention_mask = tokenized['attention_mask']

    # 3. 라벨 생성
    labels = torch.full_like(input_ids, fill_value=-100)

    assistant_header = '<|start_header_id|>assistant<|end_header_id|>\n'
    assistant_token_id = tokenizer.encode(assistant_header, add_special_tokens=False)
    eot_token = '<|eot_id|>'
    eot_token_id = tokenizer.encode(eot_token, add_special_tokens=False)

    for i, ids in enumerate(input_ids):
        ids_list = ids.tolist()
        start = None
        for idx in range(len(ids_list) - len(assistant_token_id) + 1):
            if ids_list[idx:idx+len(assistant_token_id)] == assistant_token_id:
                start = idx + len(assistant_token_id)
                break

        if start is not None:
            end = None
            for idx in range(start, len(ids_list) - len(eot_token_id) + 1):
                if ids_list[idx:idx+len(eot_token_id)] == eot_token_id:
                    end = idx + len(eot_token_id)
                    break

        labels[i, start:end] = input_ids[i, start:end]

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels
    }

# data_collator([train_data[0], train_data[1]])

NameError: name 'tokenizer' is not defined

In [ ]:
example = train_data[256]
batch = data_collator([example])      # 배치 1개로 콜레이터 적용
print(f'{batch['input_ids'].shape}')  # input_ids 텐서 shape
print(f'{batch['attention_mask'].shape}')  # attention_mask 텐서 shape
print(f'{batch['labels'].shape}')     # labels 텐서 shape

In [ ]:
print(batch['input_ids'][0].tolist())  # 0번째 샘플 input_ids 리스트
print(batch['attention_mask'][0].tolist())  # 0번째 샘플 attention_mask 리스트
print(batch['labels'][0].tolist())     # 0번째 샘플 labels 리스트 (답변 제외 -100)

In [ ]:
label_ids = [token_id for token_id in batch['labels'][0].tolist() if token_id != -100]
text = tokenizer.decode(label_ids)  # 문자열로 디코딩
text

In [ ]:
text_tokens = []  # 토큰 ID를 디코딩한 문자열을 담을 리스트
for i, token_id in enumerate(batch['input_ids'][0].tolist()):  # 토큰 ID 순회
    decoded_str = tokenizer.decode([token_id])  # 토큰 1개를 문자열로 디코딩
    text_tokens.append(decoded_str)  # 디코딩 결과 누적

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'token': text_tokens,  # 토큰 (디코딩된 문자열)
    'input_ids': batch['input_ids'][0].tolist(),  # 입력 토큰 ID
    'attention_mask': batch['attention_mask'][0].tolist(),  # 패딩 여부(1/0)
    'labels': batch['labels'][0].tolist()  # 정답 라벨 (마스킹: -100)
}).transpose()  # (행=토큰 위치)

pd.set_option('display.max_columns', None)  # 컬럼 생략 없음

In [ ]:
from peft import LoraConfig, get_peft_model  # LoRA 설정 / 적용 함수

lora_config = LoraConfig(
    r = 8,               # 저랭크 행렬 rank
    lora_alpha = 32,     # LoRA 스케일 계수 (alpha/r)
    lora_dropout = 0.1,
    bias = "none",       # bias 학습 제외
    target_modules = ['q_proj', 'v_proj'],  # LoRA를 주입할 모듈
    task_type = "CAUSAL_LM"  # 작업 유형 : 생성형
)

model = get_peft_model(model, lora_config)  # 기존 모델에 LoRA 어댑터 적용
model.print_trainable_parameters()  # 학습 가능한 파라미터 수 (비율)

In [ ]:
from trl import SFTConfig  # TRL SFT 학습 설정 클래스

hub_model_id = 'capybaraOh/Llama-VARCO-8b-news2stock-analyzer'  # 학습 완료 후 업로드할 Hub 모델 ID

sft_config = SFTConfig(  # SFT 학습 하이퍼파라미터/저장/로그 설정
    output_dir="Llama-VARCO-8b-news2stock-analyzer", # 학습 완료된 모델과 체크포인트가 저장될 경로이다.
    num_train_epochs=3,                              # 전체 데이터셋을 반복 학습할 횟수(Epoch)이다.
    per_device_train_batch_size=2,                   # 각 GPU(장치)당 한 번에 처리할 데이터 샘플의 개수이다.
    gradient_accumulation_steps=2,                   # 그래디언트를 2번 누적한 후 가중치를 업데이트한다. (실제 배치 크기 = 2 * 2 = 4 효과를 낸다.)
    gradient_checkpointing=True,                     # VRAM 절약을 위해 중간 활성화 값을 저장하지 않고 역전파 시 재계산하는 설정이다.
    optim="adamw_torch_fused",                       # 최적화 알고리즘 설정이다. fused 버전은 CUDA에서 더 빠르다.
    logging_steps=10,                                # 10 스텝마다 학습 로그(Loss 등)를 출력한다.
    save_strategy="steps",                           # 체크포인트 저장 기준을 'steps'(스텝 수)로 설정한다. (옵션: 'epoch')
    save_steps=50,                                   # 50 스텝마다 모델 체크포인트를 저장한다.
    bf16=True,                                       # BF16(Brain Float 16) 정밀도를 사용하여 메모리를 아끼고 연산 속도를 높인다. (Ampere GPU 이상 권장)
    learning_rate=1e-4,                              # 학습률(Learning Rate)이다. 가중치 업데이트의 크기를 결정한다.
    max_grad_norm=0.3,                               # 그래디언트 클리핑 임계값이다. 그래디언트 폭주를 막아 학습 안정성을 높인다.
    warmup_steps=0.03,                               # 전체 학습 단계의 3% 동안 학습률을 서서히 올리는 웜업(Warmup)을 수행한다.
    lr_scheduler_type="constant_with_warmup",        # 학습률 스케줄러 타입이다. 여기서는 학습률을 서서히 올려준다.
    push_to_hub=True,                                # 학습이 끝나면 Hugging Face Hub에 모델을 자동으로 업로드한다.
    hub_model_id=hub_model_id,                       # Hub에 업로드될 때 사용될 저장소(Repository) ID이다.
    hub_token=True,                                  # Hub 업로드를 위해 인증 토큰을 사용한다.
    remove_unused_columns=False,                     # 데이터셋에서 모델의 forward 메서드 시그니처에 없는 컬럼을 자동으로 삭제하지 않도록 한다.
    dataset_kwargs={"skip_prepare_dataset": True},   # 데이터셋 처리 과정(packing 등)을 건너뛰도록 하는 설정이다.
    report_to=['wandb'],                                    # 학습 기록을 전송할 툴(WandB, Tensorboard 등)을 지정한다. 빈 리스트는 기록하지 않음을 의미한다.
    label_names=["labels"],                          # 손실(Loss) 계산 시 정답(Target)으로 사용할 데이터셋의 컬럼 이름이다.
)

In [ ]:
from trl import SFTTrainer  # SFT 학습용 Trainer

trainer = SFTTrainer(
    model = model,      # LoRA 적용된 학습 모델
    args = sft_config,  # SFT 학습 설정
    train_dataset = train_data,  # 학습 데이터셋
    data_collator = data_collator   # 배치 텐서 생성 함수
)

trainer.train()  # 학습 실행

In [ ]:
prompt_list = []  # 프롬프트 (assistant 답변 내용 이전) 리스트
label_list = []   # 정답(assistant 답변 내용) 리스트

for messages in test_data["messages"]:
    # 채팅 템플릿 문자열로 반환
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    # assistant 답변 내용 전까지 input
    input = text.split('<|start_header_id|>assistant<|end_header_id|>\n')[0] + '<|start_header_id|>assistant<|end_header_id|>\n'
    # assistant 답변 내용 (종료 토큰 전) 추출
    labels = text.split('<|start_header_id|>assistant<|end_header_id|>\n')[1].split('<|eot_id|>')[0]
    prompt_list.append(input)
    label_list.append(labels)

In [ ]:
prompt_list[100]

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, pipeline  # 토크나이저 자동 로더 / 파이프라인 생성
import torch

peft_model_name = hub_model_id  # 업로드 된 PEFT 모델 repo

# 파인튜닝 된 PEFT 모델 로드
finetuned_model = AutoPeftModelForCausalLM.from_pretrained(
    peft_model_name,
    dtype = torch.bfloat16,  # 가중치 로딩 dtype(bf16)
    device_map = 'auto',      # 환경에 맞춰 CPU / GPU 자동 배치
    quantization_config=quant_config
)

tokenizer = AutoTokenizer.from_pretrained(peft_model_name)  # 같은 repo에서 토크나이저 로드
# 텍스트 생성 파이프라인
pipe = pipeline('text-generation', model=finetuned_model, tokenizer=tokenizer)
pipe

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, pipeline

merged_model_id = 'capybaraOh/Llama-VARCO-8b-news2stock-analyzer-4bit-merged'

tokenzier = AutoTokenizer.from_pretrained(peft_model_name)
finetuned_model = AutoPeftModelForCausalLM.from_pretrained(
    peft_model_name,
    dtype=torch.bfloat16,
    device_map='auto',
    quantization_config=quant_config
)

merged_model = finetuned_model.merge_and_unload()
merged_model.push_to_hub(merged_model_id, token=True)
tokenizer.push_to_hub(merged_model_id, token=True)

In [ ]:
del finetuned_model, pipe, merged_model

import gc
gc.collect()

torch.cuda.empty_cache()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = 'capybaraOh/Llama-VARCO-8b-news2stock-analyzer-4bit-merged'

finetuned_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map='auto'
)

tokenizer = AutoTokenizer.from_pretrain(model_id)
pipe = pipeline('text-generation', model=finetuned_model, tokenizer=tokenizer)
pipe

In [ ]:
eos_token = tokenizer('<|eot_id|>', add_special_tokens=False)['input_ids'][0]

def test_inference(pipe, prompt):
    # 파이프라인으로 결정론적인 답변 생성
    outputs = pipe(prompt, max_new_tokens=1024, eos_token_id=eos_token, do_sample=False)
    assistant_start = len(prompt)  # 입력 구간 이후 생성 결과만 사용
    return outputs[0]['generated_text'][assistant_start:].strip()  # 프롬프트 이후(생성된 부분)만 반환

In [ ]:
for prompt, label in zip(prompt_list[10:13], label_list[10:13]):  # 10 ~ 12 샘플
    print(f"[prompt] : {prompt}")
    print(f"[label] : {label}")
    print(f"[response] : {test_inference(pipe, prompt)}")
    print('='*100)

In [ ]:
def inference(news):
    messages = [
        {'role': 'system', 'content': '''
당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.

다음 출력지시사항을 지켜주세요.
1. 뉴스와 종목간의 연관성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 연관성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.
'''},
        {'role': 'user', 'content': news}
    ]

    # messages를 채팅 프롬프트 문자열로 반환
    prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    outputs = pipe(prompt, max_new_tokens=1024, eos_token_id=eos_token, do_sample=False)
    assistant_start = len(prompt)  # 입력 구간 이후 생성 결과만 사용
    return outputs[0]['generated_text'][assistant_start:].strip()

In [ ]:
news = '''
엔화, 반년 만에 최강세…"추가 상승 가능성 크다"

달러당 153엔대까지 상승
美日 추가 개입 가능성
금리인상 전망 등 영향
일본 엔화 가치가 달러당 153엔대까지 가파르게 상승했다. 지난 2월 중순 이후 반년 만에 가장 높은 수준이다. 일본 정부와 일본은행(BOJ)의 외환시장 개입 당시에도 넘지 못했던 달러당 155엔 선을 돌파한 이후에도 오름세가 이어지고 있다. 올해 고점인 152엔대까지 상승할 여지가 있다는 관측이 나온다.

8일 니혼게이자이신문(닛케이)에 따르면 이날 오전 도쿄 외환시장에서 엔화 가치는 장중 달러당 153엔대까지 뛰었다. 전날 달러당 155엔 선을 넘어섰으나, 이후에도 매수세가 지속됐다. 달러당 153엔대는 지난 2월 중순 이후 최고 수준이다. 지난 4~5월과 7월, 일본 당국이 외환시장 개입에 나섰을 당시에도 이 수준은 넘지 못했다.

이번 엔화 강세에는 미국과 일본 통화당국이 추가로 외환시장에 개입할 수 있다는 가능성과, BOJ의 금리 인상 가속화 전망 등이 복합적으로 영향을 미친 것으로 보인다고 닛케이는 짚었다.

먼저 외환시장에서는 이달 이후 미·일 통화당국이 엔화 약세 시정을 위한 구체적인 조치를 취할 것이라는 전망이 나오고 있다. 스콧 베선트 미국 재무부 장관은 지난달 30일 우에다 가즈오 BOJ 총재와 만나 "엔화의 대폭적인 저평가에 대처하기 위해 일본이 단호한 시장·금융 정책상 조치를 취하는 것을 강력히 지지한다"고 발언한 바 있다. 가타야마 사쓰키 일본 재무상도 과도한 엔화 약세가 이어질 경우 미국과 협조해 추가 개입에 나설 것을 시사해왔다.

BOJ의 금리 인상 속도가 빨라질 것이라는 전망도 엔화 매수세를 부추기고 있다. 시장에서는 BOJ가 오는 17~18일 열리는 금융정책결정회의에서 기준금리를 0.25%포인트 인상할 가능성을 선반영하고 있다. 여기에 시장은 추가 금리 상승 가능성에 베팅하고 있다. 닛케이는 "이번 기준금리 인상 이후에도 3개월에 한 번 정도의 속도로 금리 인상을 이어가거나, 최종 기준금리 목표치가 상향 조정될 것이라는 전망이 확산하고 있다"고 전했다.

여기에 중동 정세 긴장이 완화될 것이라는 기대감도 엔화 강세에 힘을 보태고 있다. 에스마일 바가이 이란 외무부 대변인은 전날 호르무즈 해협의 임시 항로를 둘러싼 오만과의 협상이 최종 단계에 도달했으며, 빠르면 며칠 내로 합의에 이를 전망이라고 밝힌 바 있다. 이로 인해 안전자산 선호 현상으로 나타났던 '유사시 달러 매수 현상'도 주춤해진 분위기다.

엔화 강세가 나타나면서 엔저에 베팅하던 투자자들의 포지션 청산도 가속화되고 있다. 엔화 강세로 손실이 커지자, 달러를 팔고 엔화를 되사들이면서 상승세를 더 부추기는 모습이다. 미쓰비시UFJ신탁은행 자금·외환부의 오카다 유스케 상급조사역은 "헤지펀드(같은 단기 투기 세력)뿐만 아니라 중장기적 관점을 가지고 거래하는 주체들도 엔 매도·달러 매수 포지션을 청산하는 등, 최근 추세가 변화하고 있다"고 닛케이에 전했다.

여기에 달러당 155엔 선이 무너지면서 손절매까지 잇따랐다. 블룸버그통신은 익명의 트레이더를 인용해 155엔 아래에 설정돼 있던 대규모 손절매 주문이 엔화가 상승하며 실행됐고, 옵션 딜러들도 달러 매도에 나서면서 상승세에 힘을 실었다고 분석했다.

이렇게 복합적인 요인들이 겹치면서 이번 엔화 급상승은 지난번 당국의 직접적인 외환 시장 개입에 따른 상승과는 성격이 다르다는 평가도 나온다. 반 루 러셀인베스트먼츠 글로벌 채권·외환 솔루션 전략 책임자는 "첫 번째 시장 개입의 효과는 이미 사라진 것으로 보인다. 이번 두 번째 상승은 시장 자체의 힘으로 나타나는 것으로 보고, 그렇기에 이번 움직임이 훨씬 더 중요하다"고 블룸버그에 전했다.

추가 상승 여력이 있다는 전망도 나왔다. 우에노 다이사쿠 미쓰비시UFJ·모건스탠리증권 수석 외환전략가는 "심리적 저항선으로 볼 수 있는 155엔을 넘어섰기에 단기적으로는 엔화 매수세가 유입되기 쉽다"고 닛케이에 전했다. 그러면서 "당분간은 올해 고점인 152엔대까지 상승 여지가 있을 것"이라고 덧붙였다.
'''

inference(news)

In [ ]:
base_model_id = 'NCSOFT/Llama-VARCO-8B-Instruct'  # 사전학습 모델명

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    dtype = torch.bfloat16,  # 가중치 로딩 dtype(bf16)
    device_map = 'auto'      # 환경에 맞춰 CPU / GPU 자동 배치
)

base_pipe = pipeline('text-generation', model=base_model, tokenizer=tokenizer)

# 베이스 모델과 LoRA 파인튜닝 모델의 응답과 정답 비교
for idx, (prompt, label) in enumerate(zip(prompt_list[10:13], label_list[10:13])):
    print(f"[샘플 {idx + 1}]")
    base_resp = test_inference(base_pipe, prompt)
    lora_resp = test_inference(pipe, prompt)
    print(f"[Base - 파인튜닝 전] {base_resp}")
    print(f"[LoRA - 파인튜닝 후] {lora_resp}")
    print(f"[Label] {label}")
    print("=" * 100)